# Distributed data: ShardedByS3Key vs FullyReplicated (SageMaker SDK v3)

Changes vs. the v2 version:
- `get_execution_role` now comes from `sagemaker.core.helper.session_helper`.
- `sagemaker.amazon.common` is gone in v3, so the data is uploaded as **headerless CSV, label in the first column**, which Linear Learner accepts natively (`ContentType: text/csv`).
- Training jobs are launched with boto3 `create_training_job`, which works regardless of SDK version.

In [ ]:
# S3 bucket and prefix
bucket = 'solutionudacity'
prefix = 'sagemaker/DEMO-data-distribution-types'

# Define IAM role (SDK v3 location, with a v2 fallback)
import boto3
import re
try:
    from sagemaker.core.helper.session_helper import get_execution_role
except ImportError:
    from sagemaker import get_execution_role

role = get_execution_role()
print(role)

In [ ]:
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import io
import time
import copy
import json
import sys
import os
# import sagemaker.amazon.common as smac   # removed in SDK v3 -> replaced by CSV upload below

In [ ]:
def get_gdelt(filename):
    s3 = boto3.resource('s3')
    s3.Bucket('gdelt-open-data').download_file('events/' + filename, '.gdelt.csv')
    # GDELT event files have no header row -> header=None so the first record isn't swallowed
    df = pd.read_csv('.gdelt.csv', sep='\t', header=None, low_memory=False)
    header = pd.read_csv('https://www.gdeltproject.org/data/lookups/CSV.header.historical.txt', sep='\t')
    df.columns = header.columns
    return df

In [ ]:
data = get_gdelt('1979.csv')
data

In [ ]:
data = data[['EventCode', 'NumArticles', 'AvgTone', 'Actor1Geo_Lat', 'Actor1Geo_Long', 'Actor2Geo_Lat', 'Actor2Geo_Long']].copy()
data['EventCode'] = data['EventCode'].astype(object)

for column in data.select_dtypes(include=['object']).columns:
    display(pd.crosstab(index=data[column], columns='% observations', normalize='columns'))

display(data.describe())
hist = data.hist(bins=30, sharey=True, figsize=(10, 10))
plt.show()

In [ ]:
events = pd.crosstab(index=data['EventCode'], columns='count').sort_values(by='count', ascending=False).index[:20]

## Similarly doing for other years

In [ ]:
s3_client = boto3.client('s3')


def write_to_s3(bucket, prefix, channel, file_prefix, X, y):
    """Replacement for smac.write_numpy_to_dense_tensor.

    Writes a headerless CSV with the label in the first column,
    the CSV layout Linear Learner expects for training.
    """
    data = np.column_stack([y.astype('float32'), X.astype('float32')])
    buf = io.BytesIO()
    np.savetxt(buf, data, delimiter=',', fmt='%.6g')
    buf.seek(0)
    key = os.path.join(prefix, channel, file_prefix + '.csv')
    s3_client.upload_fileobj(buf, bucket, key)
    print(f's3://{bucket}/{key}  ({data.shape[0]} rows, {data.shape[1] - 1} features)')


def transform_gdelt(df, events=None):
    df = df[['AvgTone', 'EventCode', 'NumArticles', 'Actor1Geo_Lat', 'Actor1Geo_Long', 'Actor2Geo_Lat', 'Actor2Geo_Long']].copy()
    if events is not None:
        df = df[np.isin(df['EventCode'], events)]          # np.in1d is deprecated in NumPy 2
        # Fixed category list -> every year gets the same dummy columns (keeps feature_dim stable)
        df['EventCode'] = pd.Categorical(df['EventCode'], categories=list(events))
    else:
        df['EventCode'] = df['EventCode'].astype(object)
    df = df[((df['Actor1Geo_Lat'] == 0) & (df['Actor1Geo_Long'] == 0) != True) &
            ((df['Actor2Geo_Lat'] == 0) & (df['Actor2Geo_Long'] == 0) != True)]
    return pd.get_dummies(df).astype('float32')             # pandas 2 returns bool dummies -> cast


def prepare_gdelt(bucket, prefix, file_prefix, events=None, random_state=1729):
    df = get_gdelt(file_prefix + '.csv')
    model_data = transform_gdelt(df, events)
    train_data, validation_data = np.split(model_data.sample(frac=1, random_state=random_state).values,
                                           [int(0.9 * len(model_data))])

    # column 0 = AvgTone (label), remaining columns = features
    write_to_s3(bucket, prefix, 'train', file_prefix, train_data[:, 1:], train_data[:, 0])
    write_to_s3(bucket, prefix, 'validation', file_prefix, validation_data[:, 1:], validation_data[:, 0])
    return model_data.shape[1] - 1   # number of features

In [ ]:
for year in range(1979, 1984):
    feature_dim = prepare_gdelt(bucket, prefix, str(year), events)

print('feature_dim =', feature_dim)

In [ ]:
region = boto3.Session().region_name

# SDK v3 image lookup, falling back to the hard-coded registry list
try:
    from sagemaker.core import image_uris
    container = image_uris.retrieve('linear-learner', region)
except Exception:
    containers = {'us-west-2': '174872318107.dkr.ecr.us-west-2.amazonaws.com/linear-learner:latest',
                  'us-east-1': '382416733822.dkr.ecr.us-east-1.amazonaws.com/linear-learner:latest',
                  'us-east-2': '404615174143.dkr.ecr.us-east-2.amazonaws.com/linear-learner:latest',
                  'eu-west-1': '438346466558.dkr.ecr.eu-west-1.amazonaws.com/linear-learner:latest'}
    container = containers[region]

print(container)

In [ ]:
common_training_params = {
    "RoleArn": role,
    "AlgorithmSpecification": {
        "TrainingImage": container,
        "TrainingInputMode": "File"
    },
    "ResourceConfig": {
        "InstanceCount": 5,
        "InstanceType": "ml.c4.2xlarge",  # ml.c5.2xlarge if c4 isn't available in your account/quota
        "VolumeSizeInGB": 10
    },
    "InputDataConfig": [
        {
            "ChannelName": "train",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": "s3://{}/{}/train/".format(bucket, prefix)
                }
            },
            "ContentType": "text/csv",       # was implicit RecordIO-protobuf before
            "CompressionType": "None",
            "RecordWrapperType": "None"
        },
        {
            "ChannelName": "validation",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": "s3://{}/{}/validation/".format(bucket, prefix),
                    "S3DataDistributionType": "FullyReplicated"
                }
            },
            "ContentType": "text/csv",
            "CompressionType": "None",
            "RecordWrapperType": "None"
        }
    ],
    "OutputDataConfig": {
        "S3OutputPath": "s3://{}/{}/".format(bucket, prefix)
    },
    "HyperParameters": {
        "feature_dim": str(feature_dim),   # 25 with the top-20 events
        "mini_batch_size": "500",
        "predictor_type": "regressor",
        "epochs": "2",
        "num_models": "32",
        "loss": "absolute_loss"
    },
    "StoppingCondition": {
        "MaxRuntimeInSeconds": 60 * 60
    }
}

In [ ]:
sharded_job = 'DEMO-linear-sharded-' + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())

print("Job name is:", sharded_job)

sharded_training_params = copy.deepcopy(common_training_params)
sharded_training_params['TrainingJobName'] = sharded_job
sharded_training_params['InputDataConfig'][0]['DataSource']['S3DataSource']['S3DataDistributionType'] = 'ShardedByS3Key'

In [ ]:
replicated_job = 'DEMO-linear-replicated-' + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())

print("Job name is:", replicated_job)

replicated_training_params = copy.deepcopy(common_training_params)
replicated_training_params['TrainingJobName'] = replicated_job
replicated_training_params['InputDataConfig'][0]['DataSource']['S3DataSource']['S3DataDistributionType'] = 'FullyReplicated'

In [ ]:
%%time

sm = boto3.client('sagemaker', region_name=region)

# Launch both jobs in parallel
sm.create_training_job(**sharded_training_params)
sm.create_training_job(**replicated_training_params)

# Wait for BOTH jobs (the original only waited on the replicated one)
waiter = sm.get_waiter('training_job_completed_or_stopped')
for job_name in [sharded_job, replicated_job]:
    waiter.wait(TrainingJobName=job_name)
    desc = sm.describe_training_job(TrainingJobName=job_name)
    status = desc['TrainingJobStatus']
    print(f"{job_name} ended with status: {status}")
    if status == 'Failed':
        print('Training failed with the following error: {}'.format(desc['FailureReason']))
        raise Exception('Training job failed')

In [ ]:
# Compare the two distribution strategies
for name, job in [('Sharded', sharded_job), ('Replicated', replicated_job)]:
    d = sm.describe_training_job(TrainingJobName=job)
    print(f"{name}: {d['TrainingJobStatus']} | billable s: {d.get('BillableTimeInSeconds')}")
    for m in d.get('FinalMetricDataList', []):
        print('   ', m['MetricName'], round(m['Value'], 4))